# `HealPixDecomp`: local multiscale decomposition

This notebook validates the exactly reconstructing Laplacian pyramid on an irregular NESTED HEALPix domain. The artificial mask mimics an ocean/land coastline. The test covers `Jmax=-1`, native-resolution metadata, exact inversion, explicit fine-grid summation, NumPy interoperability, and Torch autograd.

At this stage, `u` and `v` are decomposed as independent channels. Scale-dependent divergence/curl operators will be built on top of these bands in a subsequent step.

In [ ]:
import time

import healpix_geo
import matplotlib.pyplot as plt
import numpy as np
import torch

from healpix_analyse import HealPixDecomp

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Irregular masked domain

We begin with a spherical cap, remove a curved synthetic continent, and shuffle the remaining cell IDs. Shuffling is intentional: `invert` must return samples in exactly this original order.

In [ ]:
LEVEL = 5
CENTRE = (15.0, 25.0)
RADIUS_DEG = 35.0

candidate_ids, _, _ = healpix_geo.nested.cone_coverage(
    CENTRE, RADIUS_DEG, LEVEL, ellipsoid="sphere"
)
candidate_ids = np.asarray(candidate_ids, dtype=np.int64)
lon, lat = healpix_geo.nested.healpix_to_lonlat(
    candidate_ids.tolist(), LEVEL, ellipsoid="sphere"
)
lon = np.asarray(lon, dtype=np.float64)
lat = np.asarray(lat, dtype=np.float64)

# Curved coastline plus a small island: True means land/masked.
coast = lat > (17.0 + 8.0 * np.sin(np.deg2rad(2.5 * lon)))
island = ((lon + 5.0) / 5.0) ** 2 + ((lat - 2.0) / 4.0) ** 2 < 1.0
ocean = ~(coast | island)
cell_ids = candidate_ids[ocean]

rng = np.random.default_rng(42)
cell_ids = cell_ids[rng.permutation(len(cell_ids))]
lon, lat = healpix_geo.nested.healpix_to_lonlat(
    cell_ids.tolist(), LEVEL, ellipsoid="sphere"
)
lon = np.asarray(lon, dtype=np.float64)
lat = np.asarray(lat, dtype=np.float64)

plt.figure(figsize=(7, 5))
plt.scatter(lon, lat, c=lat, s=8, cmap="viridis")
plt.xlabel("longitude [deg]")
plt.ylabel("latitude [deg]")
plt.title(f"Masked ocean-like domain: {len(cell_ids)} cells at level {LEVEL}")
plt.colorbar(label="latitude [deg]")
plt.tight_layout()

## Synthetic velocity field and full-depth pyramid

`Jmax=-1` performs every possible Down operation, here from level 5 to level 0. Therefore the result contains five detail maps and one final coarse map.

In [ ]:
lon_rad = torch.as_tensor(np.deg2rad(lon), dtype=torch.float64, device=device)
lat_rad = torch.as_tensor(np.deg2rad(lat), dtype=torch.float64, device=device)
u = torch.cos(lat_rad) * torch.sin(2.0 * lon_rad) + 0.15 * torch.cos(5.0 * lat_rad)
v = torch.sin(lat_rad) * torch.cos(3.0 * lon_rad) - 0.10 * torch.sin(4.0 * lon_rad)
velocity = torch.stack((u, v), dim=0).unsqueeze(0)  # [B=1, C=2, N]

t0 = time.perf_counter()
decomp = HealPixDecomp(
    level=LEVEL,
    cell_ids=cell_ids,
    Jmax=-1,
    ellipsoid="sphere",
    dtype=torch.float64,
    device=device,
)
build_seconds = time.perf_counter() - t0
pyramid = decomp.compute(velocity)

print(decomp)
print("operator build [s]:", build_seconds)
print("levels:", pyramid.levels)
print("sizes :", [band.shape[-1] for band in pyramid])
assert decomp.n_scales == LEVEL
assert len(pyramid) == LEVEL + 1
assert pyramid.levels == tuple(range(LEVEL, -1, -1))
for band, ids in zip(pyramid, pyramid.cell_ids):
    assert band.shape[:-1] == velocity.shape[:-1]
    assert band.shape[-1] == len(ids)

## Exact inversion and exact cell ordering

The same paired Up prediction is subtracted during analysis and added during synthesis. Reconstruction is consequently an algebraic identity up to floating-point round-off, even though Down and Up are not mutual inverses.

In [ ]:
reconstructed = decomp.invert(pyramid)
error = reconstructed - velocity
max_abs = error.abs().max().item()
rel_rms = (
    error.square().mean().sqrt()
    / velocity.square().mean().sqrt().clamp_min(1e-15)
).item()
print({"max_abs": max_abs, "relative_rms": rel_rms})
assert reconstructed.shape == velocity.shape
assert torch.allclose(reconstructed, velocity, rtol=1e-11, atol=1e-12)

## Direct sum of fine-resolution components

The compact bands live on different native grids and cannot be added directly. `expand` lifts each band through the required paired Up operators. Their sum then equals the input.

In [ ]:
components = decomp.expand(pyramid)
direct_sum = torch.stack(components, dim=0).sum(dim=0)
sum_max_abs = (direct_sum - velocity).abs().max().item()
print({"expanded_components": len(components), "sum_max_abs": sum_max_abs})
assert len(components) == len(pyramid)
assert torch.allclose(direct_sum, velocity, rtol=1e-11, atol=1e-12)

## Native-scale bands

Each panel uses the exact NESTED IDs stored with its band. This is the metadata that later scale-dependent divergence/curl filters will consume.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
for index, (ax, band, ids, level) in enumerate(
    zip(axes.ravel(), pyramid, pyramid.cell_ids, pyramid.levels)
):
    band_lon, band_lat = healpix_geo.nested.healpix_to_lonlat(
        ids.tolist(), level, ellipsoid="sphere"
    )
    values = band[0, 0].detach().cpu().numpy()
    artist = ax.scatter(band_lon, band_lat, c=values, s=15, cmap="coolwarm")
    kind = "coarse" if index == len(pyramid) - 1 else "detail"
    ax.set_title(f"{kind}, level={level}, N={len(ids)}")
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    fig.colorbar(artist, ax=ax, shrink=0.75)

## Constant field on the masked domain

With the default L1-normalised local Down filter and column-normalised Up, a constant field should have negligible detail coefficients, including along the artificial coastline. Exact reconstruction is the strict invariant.

In [ ]:
constant = torch.ones((1, 1, len(cell_ids)), dtype=torch.float64, device=device)
constant_pyramid = decomp.compute(constant)
constant_rec = decomp.invert(constant_pyramid)
detail_max = [band.abs().max().item() for band in constant_pyramid.details]
print("detail maxima:", detail_max)
print("reconstruction max abs:", (constant_rec - constant).abs().max().item())
assert torch.allclose(constant_rec, constant, rtol=1e-11, atol=1e-12)

## Differentiability

Analysis and synthesis use Torch sparse matrix operations and preserve the autograd graph.

In [ ]:
learnable = velocity.detach().clone().requires_grad_(True)
learnable_pyramid = decomp.compute(learnable)
loss = sum(band.square().mean() for band in learnable_pyramid)
loss.backward()
assert learnable.grad is not None
assert torch.isfinite(learnable.grad).all()
print({"loss": loss.item(), "gradient_norm": learnable.grad.norm().item()})

## NumPy and explicit `Jmax` interoperability

In [ ]:
short_decomp = HealPixDecomp(
    level=LEVEL, cell_ids=cell_ids, Jmax=2, ellipsoid="sphere",
    dtype=torch.float64, device=device,
)
numpy_velocity = velocity[0].detach().cpu().numpy()
numpy_pyramid = short_decomp.compute(numpy_velocity)
numpy_rec = short_decomp.invert(numpy_pyramid)
assert len(numpy_pyramid) == 3  # two Down operations + final coarse map
assert isinstance(numpy_rec, np.ndarray)
assert np.allclose(numpy_rec, numpy_velocity, rtol=1e-11, atol=1e-12)
print({"bands": len(numpy_pyramid), "max_abs": np.max(np.abs(numpy_rec - numpy_velocity))})

## Result

The multiscale representation is compact, local, mask-aware, differentiable, and exactly invertible. It is a Laplacian pyramid rather than an orthogonal wavelet basis: bands can be correlated and their energies need not sum to input energy. The next layer can now define physical, scale-dependent local kernels that map each `(u, v)` band to divergence and curl.